# Modality Ablation — part 1 of 2

**Which modalities are actually worth paying for, relative to the cheap
Radar + GPS configuration?**

This is the prerequisite experiment for the adaptive-routing direction: before
building a model that decides *when* to acquire an expensive modality, we need
to know what each modality is worth once the cheap ones are already in hand.

The two parts are independent and can run **in parallel on two Kaggle
sessions**. They share one protocol and write to the same output layout, so the
results merge into a single table.

| | configurations | encoder cost |
|---|---|---|
| **part 1** | GPS · GPS+Radar · GPS+Radar+Camera · full | 7.6 |
| **part 2** | GPS+Camera · GPS+LiDAR · GPS+Radar+LiDAR · GPS+Camera+LiDAR | 7.6 |

**This notebook runs part 1:** gps, gps + radar, gps + radar + camera, gps + radar + camera + lidar

Part 1 carries the key comparisons (GPS → +Radar → +Camera → full), so run it
first if you can only run one.

---

## Method

Every configuration trains **independently from scratch** under an identical
protocol — same split, samples, labels, window, seed, optimiser, schedule,
epochs, batch size and metrics. The only thing that varies is which modalities
the model may use.

This reuses AMBER's own missing-modality mechanism rather than building a
different architecture per configuration: a configuration forces the eq. (3)
availability vector, and the eq. (23)/(30) masks do the rest. Verified
property — a modality marked unavailable has **exactly zero** influence on the
logits, so the restriction is faithful rather than approximate.

The mask can only ever *remove* a modality, never add one. A sample that
genuinely lacks radar stays without radar. So "GPS + Radar" on scenario 34 is
partly "GPS only" in practice, and the notebook reports the effective
availability alongside each result so the numbers can be read honestly.

## Two methodological decisions, stated rather than buried

**Historical beam indices are disabled.** AMBER treats the previous 4 beams as
a fifth input modality (its eq. 13), but they are *not* part of the DeepSense6G
2022 challenge input specification, and they are absent for 100 % of the
official test split and 100 % of the adaptation split. Leaving them on would
let a signal that vanishes at inference dominate the ablation — AMBER's own
Table V has BeamIdx+GPS alone at 58.81 % Top-1. `USE_HISTORICAL_BEAMS` exposes
this; it is `False` for the challenge-compliant comparison.

**Modality dropout is off.** The main AMBER run trains with random modality
masking, but with only one or two modalities enabled that behaves differently
per configuration — for a single-modality config the "keep at least one"
fallback makes it a no-op. Leaving it on would vary the protocol across
configurations, which is exactly what a controlled ablation must not do.

## 1. Configuration

In [1]:
from pathlib import Path

PART = 1

# Modality configurations trained by THIS notebook. Edit freely -- a
# configuration whose result already exists is skipped, so re-running is cheap.
EXPERIMENTS = [["gps"], ["gps", "radar"], ["gps", "radar", "camera"], ["gps", "radar", "camera", "lidar"]]

# Historical beam indices are an AMBER addition, not a challenge input, and are
# absent from the entire test and adaptation splits. See the header.
USE_HISTORICAL_BEAMS = False

EPOCHS      = 10      # see the note below
BATCH_SIZE  = 16      # AMBER Table II
LR          = 1e-4    # AMBER Table II
POOL        = 4       # VA = HA
SEED        = 2022
WORKERS     = 4
MODALITY_DROPOUT = 0.0   # off for a controlled ablation -- see the header

# Where results land. Stable filenames so a later notebook can load them.
OUT = Path("/kaggle/working/outputs/modality_ablation")
RUNS = OUT / "runs"
for d in (OUT, OUT / "plots", OUT / "per_config", RUNS):
    d.mkdir(parents=True, exist_ok=True)

print(f"part {PART}: {len(EXPERIMENTS)} configurations")
for e in EXPERIMENTS:
    print("   ", " + ".join(e))
print(f"\nhistorical beams: {USE_HISTORICAL_BEAMS}   epochs: {EPOCHS}   "
      f"modality dropout: {MODALITY_DROPOUT}")

part 1: 4 configurations
    gps
    gps + radar
    gps + radar + camera
    gps + radar + camera + lidar

historical beams: False   epochs: 10   modality dropout: 0.0


### Why 10 epochs

The reference 20-epoch AMBER run peaked at epoch 13 and gained nothing over
epochs 14–20, so 10 epochs with a cosine schedule *sized for 10* anneals fully
and lands close to the plateau. That matters here because the budget is eight
configurations rather than one.

Sizing: the full configuration costs ~12 min/epoch at 8,844 training samples.
This notebook's configurations total ~7.6 encoder-cost units against the full
configuration's 3.8, so ≈ 24 min per epoch across all four, i.e. **≈ 4 h for 10
epochs** — inside Kaggle's ~9 h GPU session with margin.

Raising `EPOCHS` is scientifically preferable if you have the budget; just keep
it **identical across both parts**, or the comparison breaks.

## 2. Environment

In [2]:
import socket
import subprocess
import sys

import torch

print(f"torch {torch.__version__}   cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  {p.total_memory / 1e9:.1f} GB")
else:
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU, then Run All again.")


def has_internet(host="github.com", port=443, timeout=6):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


if not has_internet():
    raise SystemExit("Internet is off; the code cannot be cloned and pretrained "
                     "ResNet weights cannot be downloaded. Settings -> Internet -> On.")
print("internet: on")

torch 2.10.0+cu128   cuda: True
GPU: Tesla T4  15.6 GB
internet: on


## 3. Locate and stage the canonical preprocessed dataset

In [3]:
import os
import shutil

import pandas as pd

INPUT_ROOT = Path("/kaggle/input")
SOURCES = ("development", "adaptation", "test")


def walk_dirs(base, follow=True):
    for dirpath, dirnames, filenames in os.walk(base, followlinks=follow):
        yield Path(dirpath), dirnames, filenames


def find_index(base):
    for cand in (base / "index" / "index", base / "index", base):
        if (cand / "samples.csv").exists():
            return cand
    for d, _dirs, files in walk_dirs(base):
        if "samples.csv" in files:
            return d
    return None


def find_source(base, name):
    for cand in (base / name / name, base / name):
        if cand.is_dir() and any(cand.glob("scenario*")):
            return cand
    return None


INDEX_SRC = find_index(INPUT_ROOT)
if INDEX_SRC is None:
    raise SystemExit("samples.csv not found under /kaggle/input. Attach the "
                     "preprocessed dataset via + Add Input.")
CANDIDATE_ROOTS = [INDEX_SRC.parent, INDEX_SRC.parent.parent, INDEX_SRC]
SOURCE_DIRS = {s: next((d for d in (find_source(r, s) for r in CANDIDATE_ROOTS) if d), None)
               for s in SOURCES}
for s in [k for k, v in SOURCE_DIRS.items() if v is None]:
    for d, dirnames, _f in walk_dirs(INPUT_ROOT):
        if d.name == s and any(n.startswith("scenario") for n in dirnames):
            SOURCE_DIRS[s] = d
            break

STAGE = Path("/kaggle/working/data/processed_amber")
STAGE.mkdir(parents=True, exist_ok=True)
INDEX_DIR = STAGE / "index"
if not (INDEX_DIR / "samples.csv").exists():
    shutil.copytree(INDEX_SRC, INDEX_DIR, dirs_exist_ok=True)
for name, src in SOURCE_DIRS.items():
    if src is not None and not (STAGE / name).exists():
        (STAGE / name).symlink_to(src)

print(f"index: {INDEX_SRC}")
for s, d in SOURCE_DIRS.items():
    print(f"  {s:<12} {d}")
assert SOURCE_DIRS["development"], "development source not found -- training needs it"

index: /kaggle/input/datasets/rakshithsrinivasan04/deepsense-preprocessed-dataset/index/index
  development  /kaggle/input/datasets/rakshithsrinivasan04/deepsense-preprocessed-dataset/development/development
  adaptation   /kaggle/input/datasets/rakshithsrinivasan04/deepsense-preprocessed-dataset/adaptation/adaptation
  test         /kaggle/input/datasets/rakshithsrinivasan04/deepsense-preprocessed-dataset/test/test


## 4. Repository code

In [4]:
REPO = "https://github.com/SathishAdithiyaaSV/MultiModal-Beam-Prediction.git"
PROJECT = Path("/kaggle/working/project")

if not (PROJECT / "amber" / "model.py").exists():
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, str(PROJECT)],
                       capture_output=True, text=True,
                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode != 0:
        raise SystemExit(f"clone failed: {(r.stderr or '').strip()[:300]}")
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from amber.ablation import (active_parameters, config_label, effective_availability,
                            enabled_mask, measure_gflops, measure_latency,
                            parse_modalities)
from amber.config import MODALITIES, ModelConfig, TrainConfig
from amber.dataset import AmberDataset
from amber.predict import load_checkpoint
from amber.train import build_parser, make_loader, to_device, train

print(f"cwd {Path.cwd()}")
print(f"modalities in the model: {MODALITIES}")

cwd /kaggle/working/project
modalities in the model: ('image', 'lidar', 'radar', 'beam', 'gps')


## 5. The dataset this experiment consumes

Reported so the results can be read against what the model actually received.
Note the availability columns: scenario 34 has radar for only ~40 % of its
samples, and beam history is absent from the adaptation and test splits
entirely.

In [5]:
samples = pd.read_csv(INDEX_DIR / "samples.csv")
usable = samples[samples.split.isin(["train", "val", "adaptation", "test"])]

print("split sizes:")
print(samples.split.value_counts().to_string(), "\n")
print("availability by split (the eq. 3 mask):")
print(usable.groupby("split")[[f"m_{m}" for m in MODALITIES]].mean().round(3).to_string())
print("\nsamples per split and scenario:")
print(usable.groupby(["split", "scenario"]).size().to_string())

split sizes:
split
train               8844
val                 2198
test                 625
excluded_nan_pwr     101
adaptation           100 

availability by split (the eq. 3 mask):
            m_image  m_lidar  m_radar  m_beam  m_gps
split                                               
adaptation      1.0      1.0    1.000   0.000    1.0
test            1.0      1.0    1.000   0.000    1.0
train           1.0      1.0    0.777   0.979    1.0
val             1.0      1.0    0.794   0.966    1.0

samples per split and scenario:
split       scenario  
adaptation  scenario31      50
            scenario32      25
            scenario33      25
test        scenario31     300
            scenario32      54
            scenario33     140
            scenario34     131
train       scenario32    2495
            scenario33    3049
            scenario34    3300
val         scenario32     600
            scenario33     750
            scenario34     848


**Which split is reported.** `val` (2,198 labelled, scenarios 32/33/34) is the
primary result. `adaptation` (100 labelled, scenarios 31/32/33) is a secondary
check and the only source of scenario-31 numbers. `test` is **unlabelled**, so
it cannot be scored — it is excluded from this experiment entirely.

Scenario 31 therefore rests on 50 adaptation samples. That is thin, and the
notebook flags it rather than presenting those numbers as equally solid.

## 6. Modality configurations

In [6]:
def resolve(cfg_modalities):
    '''Canonical modality tuple for a configuration, honouring the beam switch.'''
    mods = list(parse_modalities(cfg_modalities))
    if USE_HISTORICAL_BEAMS and "beam" not in mods:
        mods.append("beam")
    if not USE_HISTORICAL_BEAMS and "beam" in mods:
        mods.remove("beam")
    return tuple(mods)


def slug(mods):
    return "_".join(mods)


CONFIGS = []
for spec in EXPERIMENTS:
    mods = resolve(spec)
    CONFIGS.append({"modalities": mods, "label": config_label(mods), "slug": slug(mods)})

print(f"{'label':<36}{'slug':<28}modalities")
for c in CONFIGS:
    print(f"{c['label']:<36}{c['slug']:<28}{c['modalities']}")

label                               slug                        modalities
GPS                                 gps                         ('gps',)
GPS + Radar                         gps_radar                   ('gps', 'radar')
GPS + Radar + Camera                gps_radar_image             ('gps', 'radar', 'image')
GPS + Radar + Camera + LiDAR        gps_radar_image_lidar       ('gps', 'radar', 'image', 'lidar')


## 7. Train + evaluate one configuration

Each configuration calls the repository's own `amber.train.train` with
`--modalities`, so the architecture, losses, schedule and metrics are exactly
the ones used everywhere else in the project. Nothing about the model is
reimplemented here.

`skip_unavailable_encoders` is switched on so a configuration does not push
zeros through the encoders of modalities it cannot use. Verified numerically
equivalent (Δlogits = 0.0); it exists so the measured latency, FLOPs and active
parameter count reflect what the configuration would really cost to deploy.

In [7]:
import gc
import json
import time


def cost_of(ckpt_path, mods):
    '''Active parameters, latency and FLOPs for a trained configuration.'''
    device = torch.device("cuda")
    model, mcfg = load_checkpoint(ckpt_path, device)
    ds = AmberDataset(INDEX_DIR, "val", mcfg, modalities=mods)
    loader = make_loader(ds, BATCH_SIZE, shuffle=False, workers=0)
    batch = to_device(next(iter(loader)), device)
    batch["availability"] = batch["availability"] * enabled_mask(mods, device)

    out = active_parameters(model, mods)
    out["latency_ms_per_sample"] = measure_latency(model, batch, device)
    out["gflops_per_sample"] = measure_gflops(model, batch)
    del model, loader, ds, batch
    gc.collect()
    torch.cuda.empty_cache()
    return out


def run_configuration(cfg):
    '''Train one configuration from scratch and record metrics + cost.'''
    result_path = OUT / "per_config" / f"{cfg['slug']}.json"
    if result_path.exists():
        print(f"[skip] {cfg['label']} -- result already at {result_path.name}")
        return json.loads(result_path.read_text())

    print(f"\n{'=' * 78}\n{cfg['label']}   ({', '.join(cfg['modalities'])})\n{'=' * 78}",
          flush=True)
    args = build_parser().parse_args([
        "--index-dir", str(INDEX_DIR),
        "--run-dir", str(RUNS),
        "--name", cfg["slug"],
        "--modalities", *cfg["modalities"],
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--lr", str(LR),
        "--pool", str(POOL),
        "--seed", str(SEED),
        "--workers", str(WORKERS),
        "--modality-dropout", str(MODALITY_DROPOUT),
        "--device", "cuda",
        "--skip-unavailable-encoders",
        "--resume",
        "--splits", "val", "adaptation",
    ])
    started = time.time()
    train(args)
    train_seconds = time.time() - started

    run_dir = RUNS / cfg["slug"]
    metrics = json.loads((run_dir / "metrics.json").read_text())
    history = pd.read_csv(run_dir / "history.csv")

    record = {
        "part": PART,
        "label": cfg["label"],
        "slug": cfg["slug"],
        "modalities": list(cfg["modalities"]),
        "use_historical_beams": USE_HISTORICAL_BEAMS,
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "pool": POOL,
        "seed": SEED, "modality_dropout": MODALITY_DROPOUT,
        "metrics": metrics,
        "train_seconds": train_seconds,
        "best_epoch": int(history.loc[history.val_top1.idxmax(), "epoch"]),
        "effective_availability_train": effective_availability(
            samples[samples.split == "train"], cfg["modalities"]),
        "cost": cost_of(run_dir / "best.pt", cfg["modalities"]),
    }
    result_path.write_text(json.dumps(record, indent=2))
    print(f"\nsaved -> {result_path}")

    gc.collect()
    torch.cuda.empty_cache()
    return record

## 8. Run this part's configurations

Results are written after **every** configuration, so an interrupted session
loses at most the one in flight, and re-running skips whatever finished.

In [8]:
records = []
for cfg in CONFIGS:
    records.append(run_configuration(cfg))
print(f"\n{len(records)} configurations complete for part {PART}")


GPS   (gps)
modality configuration: GPS  (gps)
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 221MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 218MB/s]


device=cuda  params=49.5M  tokens/modality={'image': 16, 'lidar': 16, 'radar': 16, 'beam': 4, 'gps': 2}  N=108
  train        8844 samples
  val          2198 samples
  adaptation    100 samples
  test          625 samples
--resume given but /kaggle/working/outputs/modality_ablation/runs/gps/last.pt not found; starting fresh
epoch   1/10  loss 9.499 (focal 0.927 cont 1.166 l2 0.0000)  val top1 0.1442 top3 0.3817 dba 0.5249  [84.6s]
epoch   2/10  loss 6.815 (focal 0.671 cont 0.531 l2 0.0000)  val top1 0.2257 top3 0.4854 dba 0.6454  [82.6s]
epoch   3/10  loss 6.405 (focal 0.631 cont 0.480 l2 0.0000)  val top1 0.2284 top3 0.5005 dba 0.6468  [82.3s]
epoch   4/10  loss 6.210 (focal 0.612 cont 0.431 l2 0.0001)  val top1 0.1561 top3 0.4163 dba 0.6175  [82.7s]
epoch   5/10  loss 6.001 (focal 0.592 cont 0.385 l2 0.0001)  val top1 0.2748 top3 0.5510 dba 0.6845  [82.3s]
epoch   6/10  loss 5.854 (focal 0.578 cont 0.371 l2 0.0002)  val top1 0.2693 top3 0.5469 dba 0.6885  [83.3s]
epoch   7/10  loss 

## 9. This part's results

A local summary so the outcome is visible immediately. The **complete
8-configuration table, the deltas against GPS + Radar, the scenario comparison,
the plots and the interpretation all live in
`modality_ablation_report.ipynb`**, which merges both parts and needs no GPU.

That separation is deliberate: the deltas are measured against GPS + Radar,
which part 1 trains, so no single training notebook can complete the
cross-part analysis on its own.

In [9]:
def flatten(r, split="val"):
    m = r["metrics"].get(split, {}).get("overall", {})
    c = r["cost"]
    return {
        "Configuration": r["label"],
        "Top-1": m.get("top1"), "Top-3": m.get("top3"),
        "Top-5": m.get("top5"), "DBA": m.get("dba"), "n": m.get("n"),
        "Params (M)": c["params_active"] / 1e6,
        "Latency (ms)": c["latency_ms_per_sample"],
        "GFLOPs": c["gflops_per_sample"],
        "Train (min)": r["train_seconds"] / 60.0,
        "Best epoch": r["best_epoch"],
    }


part_results = pd.DataFrame([flatten(r) for r in records])
part_results.to_csv(OUT / f"results_part{PART}.csv", index=False)
print(f"part {PART} -- validation split\n")
print(part_results.round(4).to_string(index=False))

part 1 -- validation split

               Configuration  Top-1  Top-3  Top-5    DBA      n  Params (M)  Latency (ms)  GFLOPs  Train (min)  Best epoch
                         GPS 0.3203 0.6142 0.7502 0.7298 2198.0      5.1690        0.4449  0.3869      14.3532          10
                 GPS + Radar 0.3521 0.6665 0.8103 0.7794 2198.0     16.5405        5.2352 23.5668      37.9882           9
        GPS + Radar + Camera 0.4431 0.8139 0.9327 0.8789 2198.0     38.0234       13.8221 71.4196      84.5721           9
GPS + Radar + Camera + LiDAR 0.4399 0.8103 0.9286 0.8759 2198.0     49.3917       19.5146 94.0856     112.6076          10


## 10. Save outputs

In [10]:
config_record = {
    "part": PART,
    "experiments": EXPERIMENTS,
    "use_historical_beams": USE_HISTORICAL_BEAMS,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "pool": POOL,
    "seed": SEED, "modality_dropout": MODALITY_DROPOUT,
    "eval_splits": ["val", "adaptation"],
    "note": ("test is unlabelled and cannot be scored; historical beams disabled "
             "as they are not a challenge input and are absent from the whole "
             "test and adaptation splits"),
}
(OUT / f"config_part{PART}.json").write_text(json.dumps(config_record, indent=2))

print(f"{OUT}:")
for p in sorted(OUT.rglob("*")):
    if p.is_file() and p.suffix in (".json", ".csv"):
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size / 1e3:.1f} KB)")

print()
print("NEXT")
print("----")
print(f"1. Save Version, then download outputs/modality_ablation/per_config/*.json "
      f"({len(records)} files from this part).")
print("2. Do the same for part 2.")
print("3. Upload all 8 json files as one Kaggle dataset, attach it to")
print("   modality_ablation_report.ipynb and Run All. That produces the complete")
print("   table, deltas, scenario comparison, plots and interpretation.")
print("   No GPU needed -- it also runs locally.")

/kaggle/working/outputs/modality_ablation:
  config_part1.json  (0.6 KB)
  per_config/gps.json  (3.2 KB)
  per_config/gps_radar.json  (3.3 KB)
  per_config/gps_radar_image.json  (3.3 KB)
  per_config/gps_radar_image_lidar.json  (3.3 KB)
  results_part1.csv  (0.7 KB)
  runs/gps/config.json  (1.1 KB)
  runs/gps/history.csv  (1.7 KB)
  runs/gps/metrics.json  (2.4 KB)
  runs/gps_radar/config.json  (1.1 KB)
  runs/gps_radar/history.csv  (1.7 KB)
  runs/gps_radar/metrics.json  (2.4 KB)
  runs/gps_radar_image/config.json  (1.1 KB)
  runs/gps_radar_image/history.csv  (1.7 KB)
  runs/gps_radar_image/metrics.json  (2.4 KB)
  runs/gps_radar_image_lidar/config.json  (1.1 KB)
  runs/gps_radar_image_lidar/history.csv  (1.7 KB)
  runs/gps_radar_image_lidar/metrics.json  (2.4 KB)

NEXT
----
1. Save Version, then download outputs/modality_ablation/per_config/*.json (4 files from this part).
2. Do the same for part 2.
3. Upload all 8 json files as one Kaggle dataset, attach it to
   modality_ablation_re